# ingest_jdbc\nGenerated from 01_ingestion/ingest_jdbc.py for Databricks notebook import.\n

In [ ]:
"""Generic JDBC ingestion adapter."""\n\nfrom __future__ import annotations\n\nimport os\n\n\ndef build_jdbc_options(jdbc_profile: dict, source: dict) -> dict:\n    """Validate and build the JDBC option dict for a Spark JDBC reader.\n\n    Credentials are read from environment variables (never hardcoded) so they\n    do not appear in source control or config files.\n    """\n    required_profile_keys = ["url", "driver", "fetchsize"]\n    missing_profile = [k for k in required_profile_keys if not jdbc_profile.get(k)]\n    if missing_profile:\n        raise ValueError(f"Missing JDBC profile fields: {missing_profile}")\n    if not source.get("jdbc_table"):\n        raise ValueError("jdbc_table is required in source metadata")\n\n    options: dict = {\n        "url": jdbc_profile["url"],\n        "dbtable": source["jdbc_table"],\n        "driver": jdbc_profile["driver"],\n        "fetchsize": str(jdbc_profile["fetchsize"]),\n    }\n\n    user_env = jdbc_profile.get("user_env", "")\n    password_env = jdbc_profile.get("password_env", "")\n    if user_env:\n        options["user"] = os.getenv(user_env, "")\n    if password_env:\n        options["password"] = os.getenv(password_env, "")\n\n    # Strip blanks so Spark doesn't receive empty-string options.\n    return {k: v for k, v in options.items() if v not in {"", None}}\n\n\ndef _apply_parallel_read_options(options: dict, extra_options: dict) -> dict:\n    """Merge parallel read options when ``numPartitions`` is configured.\n\n    For large tables, configure these in ``source_options_json``:\n\n    .. code-block:: json\n\n        {\n          "numPartitions": 8,\n          "partitionColumn": "id",\n          "lowerBound": 1,\n          "upperBound": 10000000\n        }\n\n    All four must be present together; missing any one will skip partitioning\n    and fall back to single-threaded read rather than raising an error, so\n    partial configs don't silently break things.\n    """\n    partition_keys = {"numPartitions", "partitionColumn", "lowerBound", "upperBound"}\n    present = partition_keys.intersection(extra_options)\n    if present == partition_keys:\n        for k in partition_keys:\n            options[k] = str(extra_options[k])\n    return options\n\n\ndef ingest_jdbc_batch(spark, jdbc_profile: dict, source: dict, extra_options: dict | None = None):\n    """Read a JDBC table or query into a Spark DataFrame.\n\n    Parameters\n    ----------\n    spark:\n        Active SparkSession.\n    jdbc_profile:\n        Connection profile from ``global_config.yaml`` (``connections.jdbc_profiles``).\n    source:\n        Source registry row.  Must contain ``jdbc_table``.\n    extra_options:\n        Parsed ``source_options_json`` dict.  Supports all Spark JDBC reader\n        options.  Parallel read keys (``numPartitions``, ``partitionColumn``,\n        ``lowerBound``, ``upperBound``) are handled automatically when all four\n        are present.\n\n    Returns\n    -------\n    pyspark.sql.DataFrame\n    """\n    options = build_jdbc_options(jdbc_profile, source)\n\n    # Merge parallel-read partitioning options.\n    merged_extra = dict(extra_options or {})\n    options = _apply_parallel_read_options(options, merged_extra)\n\n    # Forward any remaining extra options (e.g. queryTimeout, sessionInitStatement).\n    partition_keys = {"numPartitions", "partitionColumn", "lowerBound", "upperBound"}\n    for k, v in merged_extra.items():\n        if k not in partition_keys and isinstance(v, (str, int, float, bool)):\n            options[k] = str(v)\n\n    reader = spark.read.format("jdbc")\n    for k, v in options.items():\n        reader = reader.option(k, v)\n    return reader.load()\n